# 02 · nn.Module & the Complete Training Pipeline

| | |
|---|---|
| **Author** | Chandan C |
| **GitHub** | [github.com/chandanc5525](https://github.com/chandanc5525) |
| **Course** | Agentic AI BootCamp · Module 02 – Advanced PyTorch |
| **Repository** | [AgenticAI_BootCamp-/02_advanced_pytorch](https://github.com/chandanc5525/AgenticAI_BootCamp-/tree/main/02_advanced_pytorch) |
| **Level** | Beginner → Intermediate |
| **Prerequisites** | Notebook 01 |

---

### What you will learn
- `nn.Module`, parameters, buffers, `state_dict`
- Layers, activations, containers, initialisation
- Loss functions and optimisers, LR schedulers
- `Dataset` / `DataLoader`, collate functions, samplers
- Training / validation loops, metrics, early stopping
- Regularisation, gradient clipping, checkpointing, reproducibility

> Every cell runs on a plain CPU. If a GPU (CUDA / Apple MPS) is present, the notebooks use it automatically.

## 0. Setup

In [ ]:
import math, random, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__, device)

## 1. Your first `nn.Module`
Every model subclasses `nn.Module`: define layers in `__init__`, computation in `forward`. Assigned sub-modules and `nn.Parameter`s are registered automatically.

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

model = TinyMLP(2, 16, 3)
print(model)
for name, p in model.named_parameters():
    print(f"{name:12s} {tuple(p.shape)}  requires_grad={p.requires_grad}")
print("total params:", sum(p.numel() for p in model.parameters()))

### 1.1 Parameters, buffers, `state_dict`
*Buffers* are non-trainable tensors saved with the model (e.g. BatchNorm running stats, positional encodings, causal masks).

In [ ]:
class WithBuffer(nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(1))                 # learned
        self.register_buffer("mask", torch.tril(torch.ones(4, 4)))  # not learned, but saved
    def forward(self, x):
        return x * self.scale

m = WithBuffer()
print(list(m.state_dict().keys()))
print([n for n, _ in m.named_parameters()], [n for n, _ in m.named_buffers()])

## 2. Building blocks you will reuse in GenAI models

In [ ]:
x = torch.randn(4, 10)

# Activations
for act in [nn.ReLU(), nn.GELU(), nn.SiLU(), nn.Tanh(), nn.Sigmoid()]:
    print(f"{act.__class__.__name__:8s}", act(torch.tensor([-2., 0., 2.])).tolist())

# Normalisation: LayerNorm is the Transformer default, BatchNorm is for CNNs
ln = nn.LayerNorm(10); bn = nn.BatchNorm1d(10)
print("LayerNorm mean/std:", ln(x).mean(-1)[:2].tolist(), ln(x).std(-1, unbiased=False)[:2].tolist())

# Dropout is active only in train() mode
drop = nn.Dropout(0.5)
drop.train(); print("train:", drop(torch.ones(8)))
drop.eval();  print("eval :", drop(torch.ones(8)))

# Embedding: id -> vector (tokens, users, tools ...)
emb = nn.Embedding(num_embeddings=100, embedding_dim=8, padding_idx=0)
print(emb(torch.tensor([[1, 2, 0]])).shape)

### 2.1 Containers: `Sequential`, `ModuleList`, `ModuleDict`
A plain Python list does **not** register layers – use `ModuleList`.

In [ ]:
class FlexibleMLP(nn.Module):
    def __init__(self, dims, p=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            nn.Sequential(nn.Linear(i, o), nn.GELU(), nn.Dropout(p))
            for i, o in zip(dims[:-2], dims[1:-1])
        )
        self.head = nn.Linear(dims[-2], dims[-1])
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.head(x)

mlp = FlexibleMLP([2, 64, 64, 3])
print(sum(p.numel() for p in mlp.parameters()), "parameters")

heads = nn.ModuleDict({"intent": nn.Linear(8, 3), "sentiment": nn.Linear(8, 2)})  # multi-task heads
print({k: v(torch.randn(1, 8)).shape for k, v in heads.items()})

### 2.2 Weight initialisation

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)     # or kaiming_normal_ for ReLU nets
        nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, std=0.02)   # GPT-style
mlp.apply(init_weights)
print(mlp.head.weight.std().item())

## 3. Data: `Dataset`, `DataLoader`, splits
We create a **three-arm spiral** dataset – not linearly separable, so a non-linear network is required.

In [ ]:
def make_spirals(n_per_class=300, classes=3, noise=0.2):
    X, y = [], []
    for c in range(classes):
        r = torch.linspace(0.05, 1, n_per_class)
        t = torch.linspace(c * 4, (c + 1) * 4, n_per_class) + torch.randn(n_per_class) * noise
        X.append(torch.stack([r * torch.sin(t), r * torch.cos(t)], 1))
        y.append(torch.full((n_per_class,), c))
    return torch.cat(X), torch.cat(y)

X, y = make_spirals()
plt.figure(figsize=(4, 4)); plt.scatter(X[:, 0], X[:, 1], c=y, s=6, cmap="viridis"); plt.title("Spirals"); plt.show()

class SpiralDataset(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

full = SpiralDataset(X, y)
n_val = int(0.2 * len(full))
train_ds, val_ds = random_split(full, [len(full) - n_val, n_val], generator=torch.Generator().manual_seed(0))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False)
val_dl   = DataLoader(val_ds, batch_size=256)
xb, yb = next(iter(train_dl)); print(xb.shape, yb.shape)

### 3.1 Custom `collate_fn` – variable-length data
Real GenAI data (text) has different lengths per sample. The collate function pads a batch.

In [ ]:
from torch.nn.utils.rnn import pad_sequence

samples = [(torch.tensor([5, 6, 7]), 1), (torch.tensor([8, 9]), 0), (torch.tensor([1, 2, 3, 4]), 1)]
def collate(batch, pad_id=0):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True, padding_value=pad_id)
    attn_mask = padded != pad_id
    return padded, attn_mask, lengths, torch.tensor(labels)

for batch in DataLoader(samples, batch_size=3, collate_fn=collate):
    print(batch[0], "\n", batch[1], "\n", batch[2], batch[3])

## 4. Losses and optimisers
| Task | Loss |
|---|---|
| Multi-class classification / next-token prediction | `CrossEntropyLoss` (takes **raw logits**) |
| Binary / multi-label | `BCEWithLogitsLoss` |
| Regression | `MSELoss`, `SmoothL1Loss` |
| Embedding similarity | `CosineEmbeddingLoss`, InfoNCE |

In [ ]:
logits = torch.randn(4, 3); target = torch.tensor([0, 2, 1, 1])
print("CE            :", F.cross_entropy(logits, target).item())
print("CE (smoothing):", F.cross_entropy(logits, target, label_smoothing=0.1).item())
print("ignore_index  :", F.cross_entropy(logits, torch.tensor([0, -100, 1, 1]), ignore_index=-100).item())
print("BCE-logits    :", F.binary_cross_entropy_with_logits(torch.randn(4), torch.tensor([1., 0., 1., 0.])).item())

params = list(mlp.parameters())
sgd   = torch.optim.SGD(params, lr=0.1, momentum=0.9)
adam  = torch.optim.Adam(params, lr=1e-3)
adamw = torch.optim.AdamW(params, lr=1e-3, weight_decay=0.01)    # standard for Transformers
print(type(adamw).__name__, adamw.defaults["weight_decay"])

# Different LR / weight-decay for different parameter groups
decay    = [p for n, p in mlp.named_parameters() if p.ndim >= 2]
no_decay = [p for n, p in mlp.named_parameters() if p.ndim < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.01},
                         {"params": no_decay, "weight_decay": 0.0}], lr=3e-3)

### 4.1 Learning-rate schedulers
Warm-up + cosine decay is the default recipe for training Transformers.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, OneCycleLR

def warmup_cosine(step, warmup=20, total=200):
    if step < warmup: return (step + 1) / warmup
    progress = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1 + math.cos(math.pi * progress))

dummy = torch.optim.AdamW([nn.Parameter(torch.zeros(1))], lr=1e-3)
sched = LambdaLR(dummy, warmup_cosine)
lrs = []
for _ in range(200):
    lrs.append(dummy.param_groups[0]["lr"]); dummy.step(); sched.step()
plt.figure(figsize=(5, 2.5)); plt.plot(lrs); plt.title("Warm-up + cosine LR"); plt.xlabel("step"); plt.show()

## 5. The training loop
The canonical five steps: **forward → loss → `zero_grad` → `backward` → `step`**. We wrap it in reusable functions with validation, accuracy tracking, gradient clipping, scheduling, early stopping and best-model restore.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += loss_fn(logits, yb).item() * len(xb)
        correct += (logits.argmax(1) == yb).sum().item()
        n += len(xb)
    return total_loss / n, correct / n

def fit(model, train_dl, val_dl, epochs=60, lr=3e-3, weight_decay=1e-4,
        clip=1.0, patience=10, verbose=True):
    model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl))
    best, best_state, bad = float("inf"), None, 0
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        running, n = 0.0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)   # stabilises training
            opt.step(); sched.step()
            running += loss.item() * len(xb); n += len(xb)
        v_loss, v_acc = evaluate(model, val_dl, loss_fn)
        hist["train_loss"].append(running / n); hist["val_loss"].append(v_loss); hist["val_acc"].append(v_acc)
        if verbose and epoch % 10 == 0:
            print(f"epoch {epoch:3d} | train {running/n:.4f} | val {v_loss:.4f} | acc {v_acc:.3f}")
        if v_loss < best - 1e-4:
            best, best_state, bad = v_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                print(f"early stop at epoch {epoch}"); break
    model.load_state_dict(best_state)
    return hist

seed_everything()
model = FlexibleMLP([2, 64, 64, 3], p=0.05)
hist = fit(model, train_dl, val_dl)
print("final:", evaluate(model, val_dl, nn.CrossEntropyLoss()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(hist["train_loss"], label="train"); ax[0].plot(hist["val_loss"], label="val"); ax[0].legend(); ax[0].set_title("Loss")
ax[1].plot(hist["val_acc"]); ax[1].set_title("Val accuracy"); plt.tight_layout(); plt.show()

# decision boundary
g = torch.linspace(-1.1, 1.1, 200); gx, gy = torch.meshgrid(g, g, indexing="xy")
grid = torch.stack([gx.ravel(), gy.ravel()], 1)
model.eval()
with torch.no_grad(): pred = model(grid.to(device)).argmax(1).cpu().reshape(200, 200)
plt.figure(figsize=(4, 4)); plt.contourf(gx, gy, pred, alpha=0.3); plt.scatter(X[:, 0], X[:, 1], c=y, s=4); plt.show()

## 6. Saving and loading
- Save the **`state_dict`** (portable), not the whole pickled model.
- For resumable training also store optimiser, scheduler, epoch and RNG state.

In [ ]:
import os, tempfile
path = os.path.join(tempfile.gettempdir(), "mlp.pt")
torch.save({"model": model.state_dict(), "config": {"dims": [2, 64, 64, 3]}}, path)

ckpt = torch.load(path, map_location="cpu", weights_only=True)
restored = FlexibleMLP(ckpt["config"]["dims"]); restored.load_state_dict(ckpt["model"])
restored.eval()
with torch.no_grad():
    print("outputs identical:", torch.allclose(restored(X[:5]), model.cpu().eval()(X[:5]), atol=1e-6))
model.to(device)

## 7. Regularisation & over-fitting demo
Compare a model with and without dropout + weight-decay on a small noisy training set.

In [ ]:
seed_everything()
small_tr, small_va = random_split(full, [60, len(full) - 60], generator=torch.Generator().manual_seed(1))
sd, vd = DataLoader(small_tr, batch_size=30, shuffle=True), DataLoader(small_va, batch_size=256)
res = {}
for name, p, wd in [("no-reg", 0.0, 0.0), ("dropout+wd", 0.3, 1e-2)]:
    m = FlexibleMLP([2, 128, 128, 3], p=p)
    h = fit(m, sd, vd, epochs=80, weight_decay=wd, patience=1000, verbose=False)
    res[name] = (h["train_loss"][-1], h["val_loss"][-1], h["val_acc"][-1])
for k, v in res.items(): print(f"{k:11s} train {v[0]:.3f} | val {v[1]:.3f} | val-acc {v[2]:.3f}")

## 8. Debugging checklist
1. **Overfit one batch** - if the loss can't reach ~0, there's a bug.
2. Check shapes at every layer with a dummy forward pass.
3. Verify the loss at init: ~`ln(num_classes)` for classification.
4. Watch gradient norms; NaNs → lower LR, clip gradients, check data.

In [ ]:
m = FlexibleMLP([2, 64, 3]).to(device)
xb, yb = next(iter(train_dl)); xb, yb = xb.to(device), yb.to(device)
print("initial loss:", F.cross_entropy(m(xb), yb).item(), "| ln(3) =", math.log(3))

opt = torch.optim.Adam(m.parameters(), lr=1e-2)
for step in range(300):
    loss = F.cross_entropy(m(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
print("loss after overfitting 1 batch:", round(loss.item(), 5))

total = sum(p.grad.norm() ** 2 for p in m.parameters() if p.grad is not None) ** 0.5
print("grad norm:", float(total))
torch.autograd.set_detect_anomaly(False)   # set True to locate NaN-producing ops

## Exercises
1. Add a `BatchNorm1d` variant and compare convergence speed.
2. Implement gradient accumulation (effective batch = 4× actual) inside `fit`.
3. Create a multi-task model with two heads (`ModuleDict`) and a weighted sum of losses.
4. Write a `Dataset` that reads a CSV lazily and a collate function that pads text features.
5. Save a *resumable* checkpoint (optimiser + scheduler + epoch) and resume training.

---
### Summary & next step
Work through the exercises above before moving on. Next notebook: **03_cnn_rnn_and_sequence_models.ipynb**.

*Notebook by **Chandan C** · [github.com/chandanc5525](https://github.com/chandanc5525) · Agentic AI BootCamp*